# Hidden Markov Models (HMMs) and Viterbi Algorithm

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand Hidden Markov Models (HMMs) structure and components
- Implement HMMs for sequence prediction
- Implement Viterbi algorithm for sequence decoding
- Apply HMMs to a practical problem on **real text**: recovering a real sentence that has
  been corrupted by a noisy keyboard

## 🔗 Prerequisites

- ✅ Basic probability (see `01_learning_under_uncertainty.ipynb`)
- ✅ Markov chains — introduced from scratch in Part 0 below
- ✅ Python 3.8+ installed

---

This notebook covers practical activities from **Course 02, Unit 3**:
- Working with Hidden Markov Models (HMMs) for sequence prediction
- Implementing Viterbi algorithm for sequence decoding
- Applying HMMs to practical problems (speech recognition, POS tagging, noisy-channel decoding)

---

## Introduction to Hidden Markov Models

**Hidden Markov Models (HMMs)** are statistical models for sequences where:
- **Hidden states**: Unobserved states (e.g., weather: Sunny, Rainy)
- **Observations**: Observed outputs (e.g., activities: Walk, Shop, Clean)
- **Transitions**: Probabilities between hidden states
- **Emissions**: Probabilities of observations given states

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import numpy as np

print("✅ Libraries imported!")
print("Ready to work with HMMs and Viterbi algorithm!")


✅ Libraries imported!
Ready to work with HMMs and Viterbi algorithm!


## Part 0: A 60-Second Markov Chain Primer

An HMM is built on a **Markov chain**, so let's meet that idea first.

A Markov chain is a system that hops between **states** (e.g., Sunny / Rainy), where the probability of the next state depends ONLY on the current state — not on the full history. This is the **Markov property** ("memorylessness").

Its entire behavior is captured by a **transition matrix**: one row per current state, giving the probability of each next state (every row sums to 1).

The simulation below shows two things:
1. A sampled sequence of weather states (each day generated only from the previous day).
2. The long-run fraction of days spent in each state settles to fixed values determined by the transition matrix alone.

An HMM then adds one twist: you cannot see the states themselves — only noisy **observations** emitted from them.

In [2]:
# A tiny 2-state Markov chain: tomorrow's weather depends only on today's
np.random.seed(42)  # Reproducible simulation

# Transition matrix: P[current][next]  (each row sums to 1)
P = {
    'Sunny': {'Sunny': 0.7, 'Rainy': 0.3},
    'Rainy': {'Sunny': 0.4, 'Rainy': 0.6}
}

# Simulate 10,000 days, starting Sunny
states_list = ['Sunny', 'Rainy']
day, chain = 'Sunny', ['Sunny']
for _ in range(9999):
    probs = [P[day][s] for s in states_list]
    day = np.random.choice(states_list, p=probs)
    chain.append(day)

print("First 15 simulated days:")
print("  " + " -> ".join(s[0] for s in chain[:15]) + "   (S=Sunny, R=Rainy)")
print()

# Long-run fraction of time in each state (empirical)
frac_sunny = chain.count('Sunny') / len(chain)
frac_rainy = chain.count('Rainy') / len(chain)

# Theoretical long-run (stationary) fractions for a 2-state chain:
# pi_Sunny = P(R->S) / (P(S->R) + P(R->S))
pi_sunny = P['Rainy']['Sunny'] / (P['Sunny']['Rainy'] + P['Rainy']['Sunny'])
pi_rainy = 1 - pi_sunny

print(f"Long-run fraction of days (10,000-day simulation vs. theory):")
print(f"  Sunny: {frac_sunny:.3f}  (theory: {pi_sunny:.3f})")
print(f"  Rainy: {frac_rainy:.3f}  (theory: {pi_rainy:.3f})")
print()
print("Key point: the transition matrix alone controls the chain's behavior.")
print("Next: an HMM uses exactly this kind of chain for its HIDDEN states.")

First 15 simulated days:
  S -> S -> R -> R -> R -> S -> S -> S -> R -> R -> R -> S -> R -> R -> S   (S=Sunny, R=Rainy)

Long-run fraction of days (10,000-day simulation vs. theory):
  Sunny: 0.586  (theory: 0.571)
  Rainy: 0.414  (theory: 0.429)

Key point: the transition matrix alone controls the chain's behavior.
Next: an HMM uses exactly this kind of chain for its HIDDEN states.


## Part 1: Simple HMM Implementation

Let's implement a simple HMM for weather prediction.

> **A note on where the numbers come from.** The transition and emission tables in Parts 1–3
> are *model parameters we declare*, chosen small enough to trace by hand — the way a textbook
> states a worked example. They are not a dataset. In Part 4 we throw them away and estimate
> the whole model from **real English text**, then decode a **real sentence**.


In [3]:
# Build a Hidden Markov Model: hidden weather states we cannot see, activities we CAN observe.
# Why: HMMs answer 'what is hidden behind my observations?' - the same math behind speech recognition and tagging.
# The forward algorithm computes P(observation sequence) efficiently with dynamic programming.

class SimpleHMM:
    """Simple Hidden Markov Model implementation"""
    
    def __init__(self, states, observations, transition_probs, emission_probs, initial_probs):
        """
        Parameters:
        - states: List of hidden states
        - observations: List of possible observations
        - transition_probs: Dict of transition probabilities, indexed
          transition_probs[current_state][next_state] = P(next | current)
        - emission_probs: Dict of emission probabilities P(obs | state)
        - initial_probs: Initial state probabilities
        """
        self.states = states
        self.observations = observations
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs
        self.initial_probs = initial_probs
    
    def forward(self, obs_sequence):
        """Forward algorithm: Compute probability of observation sequence"""
        T = len(obs_sequence)
        N = len(self.states)
        
        # Initialize alpha (forward probabilities)
        alpha = np.zeros((T, N))
        
        # Initialization
        for i, state in enumerate(self.states):
            alpha[0, i] = self.initial_probs[state] * self.emission_probs[state][obs_sequence[0]]
        
        # Recursion
        for t in range(1, T):
            for j, state_j in enumerate(self.states):
                alpha[t, j] = sum(
                    alpha[t-1, i] * self.transition_probs[self.states[i]][state_j] 
                    for i in range(N)
                ) * self.emission_probs[state_j][obs_sequence[t]]
        
        # Termination
        return alpha, sum(alpha[T-1, :])

# Example: Weather HMM
states = ['Sunny', 'Rainy']
observations = ['Walk', 'Shop', 'Clean']

# Transition probabilities: P(next_state | current_state)
transition_probs = {
    'Sunny': {'Sunny': 0.7, 'Rainy': 0.3},
    'Rainy': {'Sunny': 0.4, 'Rainy': 0.6}
}

# Emission probabilities: P(observation | state)
emission_probs = {
    'Sunny': {'Walk': 0.6, 'Shop': 0.3, 'Clean': 0.1},
    'Rainy': {'Walk': 0.1, 'Shop': 0.4, 'Clean': 0.5}
}

# Initial probabilities
initial_probs = {'Sunny': 0.6, 'Rainy': 0.4}

# Create HMM
hmm = SimpleHMM(states, observations, transition_probs, emission_probs, initial_probs)

print("=" * 60)
print("Hidden Markov Model: Weather Prediction")
print("=" * 60)
print(f"States: {states}")
print(f"Observations: {observations}")

Hidden Markov Model: Weather Prediction
States: ['Sunny', 'Rainy']
Observations: ['Walk', 'Shop', 'Clean']


## Part 2: The Forward Algorithm — Probability of an Observation Sequence

Before decoding hidden states, the first classic HMM question is: **how likely is an observation sequence under this model?**

The **forward algorithm** answers it with dynamic programming. It fills a table `alpha[t, i]` = probability of seeing the first `t+1` observations AND being in state `i` at time `t`, then sums the last row.

In [4]:
# Run the forward algorithm on an observation sequence
# Why print the alpha table? Each row shows how probability mass flows through the hidden states
# one time step at a time - summing the last row gives the total sequence probability.
obs_seq = ['Walk', 'Shop', 'Clean']

alpha, total_prob = hmm.forward(obs_seq)

print("=" * 60)
print(f"Forward Algorithm: P({obs_seq})")
print("=" * 60)
print()
print(f"{'t':<4}{'observation':<14}" + "".join(f"alpha[{s}]".ljust(16) for s in hmm.states))
for t, obs in enumerate(obs_seq):
    row = "".join(f"{alpha[t, i]:<16.6f}" for i in range(len(hmm.states)))
    print(f"{t:<4}{obs:<14}" + row)
print()
print(f"P(observation sequence) = sum of last row = {total_prob:.6f}")

Forward Algorithm: P(['Walk', 'Shop', 'Clean'])

t   observation   alpha[Sunny]    alpha[Rainy]    
0   Walk          0.360000        0.040000        
1   Shop          0.080400        0.052800        
2   Clean         0.007740        0.027900        

P(observation sequence) = sum of last row = 0.035640


## Part 3: Viterbi Algorithm for Sequence Decoding

The Viterbi algorithm finds the most likely sequence of hidden states given observations.


In [5]:
# Viterbi algorithm: find the single MOST LIKELY hidden state sequence for the observations.
# Why: forward gives 'how likely are these observations overall'; Viterbi answers the decoding
# question 'what was the weather each day?' using max instead of sum, plus backpointers to recover the path.

def viterbi(hmm, obs_sequence):
    """
    Viterbi algorithm: Find most likely sequence of hidden states
    
    Returns:
    - best_path: Most likely state sequence
    - best_prob: Probability of best path
    """
    T = len(obs_sequence)
    N = len(hmm.states)
    
    # Initialize viterbi and backpointer tables
    viterbi_table = np.zeros((T, N))
    backpointer = np.zeros((T, N), dtype=int)
    
    # Initialization
    for i, state in enumerate(hmm.states):
        viterbi_table[0, i] = hmm.initial_probs[state] * hmm.emission_probs[state][obs_sequence[0]]
        backpointer[0, i] = 0
    
    # Recursion
    for t in range(1, T):
        for j, state_j in enumerate(hmm.states):
            # Find best previous state
            probs = [
                viterbi_table[t-1, i] * hmm.transition_probs[hmm.states[i]][state_j]
                for i in range(N)
            ]
            best_prev = np.argmax(probs)
            viterbi_table[t, j] = probs[best_prev] * hmm.emission_probs[state_j][obs_sequence[t]]
            backpointer[t, j] = best_prev
    
    # Termination: Find best final state
    best_final = np.argmax(viterbi_table[T-1, :])
    best_prob = viterbi_table[T-1, best_final]
    
    # Backtrack to find best path
    best_path = [hmm.states[best_final]]
    for t in range(T-1, 0, -1):
        best_final = backpointer[t, best_final]
        best_path.insert(0, hmm.states[best_final])
    
    return best_path, best_prob

# Example: Decode observation sequence
obs_seq = ['Walk', 'Shop', 'Clean']
print("=" * 60)
print(f"Observation Sequence: {obs_seq}")
print("=" * 60)

best_states, prob = viterbi(hmm, obs_seq)
print(f"Most likely state sequence: {best_states}")
print(f"Probability: {prob:.6f}")

Observation Sequence: ['Walk', 'Shop', 'Clean']
Most likely state sequence: ['Sunny', 'Rainy', 'Rainy']
Probability: 0.012960


## Part 4: Application — Recovering Real Text From a Noisy Channel

Now we run the *same* Viterbi algorithm on real data, with every number estimated rather
than invented.

**The setup.** Somebody typed a real sentence, but the keyboard was faulty: some keypresses
landed on a neighbouring key. We see only the corrupted string. What did they mean to type?

- **Hidden states** — the 26 letters plus space: the characters actually intended.
- **Observations** — the characters that came out of the faulty keyboard.
- **Transition probabilities** — estimated from **real English text**: 683,000 characters of
  genuine Usenet posts from the `sci.space` newsgroup. `P(next letter | current letter)` is
  counted from that corpus, not chosen by us. This is where the model learns that `q` is
  almost always followed by `u`.
- **Emission probabilities** — the keyboard fault model: with probability `P_ERR` the typed
  key slips to a physically adjacent key on a QWERTY layout, otherwise it is correct. This is
  noise we *deliberately inject into real text* to create the decoding problem, and we declare
  its exact parameters below.
- **Ground truth** — the real sentence, so we can *measure* how much Viterbi actually helps
  instead of just claiming it helps.

This is the same algorithm as POS tagging: swap "intended letter" for "part-of-speech tag"
and "typed character" for "word", and the code is unchanged.

In [6]:
# Estimate a REAL language model, then decode a REAL sentence through a noisy channel.
# Why real text? The transition matrix has to encode genuine English structure ("q" is
# followed by "u", "th" is common) — structure you cannot invent by hand for 27x27 = 729
# entries, and the whole demo depends on it being right.
import re
import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_20newsgroups

# --- REAL DATA: genuine Usenet posts written by real people ---
news = fetch_20newsgroups(subset="train", categories=["sci.space"],
                          remove=("headers", "footers", "quotes"), random_state=0)
raw_text = " ".join(news.data)

# Normalise to a 27-symbol alphabet: lowercase letters plus space.
corpus = re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", raw_text.lower())).strip()

ALPHABET = list("abcdefghijklmnopqrstuvwxyz ")
char_index = {c: i for i, c in enumerate(ALPHABET)}
N_STATES = len(ALPHABET)

print("=" * 60)
print("Estimating the HMM from REAL text")
print("=" * 60)
print(f"Corpus: {len(news.data)} real newsgroup posts, {len(corpus):,} characters after cleaning")

# --- TRANSITIONS: counted from the real corpus (add-one smoothing avoids zero probabilities) ---
seq = np.array([char_index[c] for c in corpus])
trans_counts = np.ones((N_STATES, N_STATES))          # start every pair at 1 = add-one smoothing
np.add.at(trans_counts, (seq[:-1], seq[1:]), 1)       # count every adjacent pair in the real text
TRANS = trans_counts / trans_counts.sum(axis=1, keepdims=True)

# --- INITIAL probabilities: how often each character appears in the real corpus ---
INITIAL = np.bincount(seq, minlength=N_STATES).astype(float)
INITIAL /= INITIAL.sum()

print("\nWhat the model learned from real English (top successor of each letter):")
for c in "qtsi ":
    label = "space" if c == " " else f"'{c}'"
    print(f"  after {label:>6}: '{ALPHABET[TRANS[char_index[c]].argmax()]}'"
          f"  (p = {TRANS[char_index[c]].max():.2f})")

# --- EMISSIONS: the declared keyboard-fault model (this is the noise we inject on purpose) ---
QWERTY_NEIGHBOURS = {
    'a': 'qwsz', 'b': 'vghn', 'c': 'xdfv', 'd': 'serfcx', 'e': 'wsdr', 'f': 'drtgvc',
    'g': 'ftyhbv', 'h': 'gyujnb', 'i': 'ujko', 'j': 'huikmn', 'k': 'jiolm', 'l': 'kop',
    'm': 'njk', 'n': 'bhjm', 'o': 'iklp', 'p': 'ol', 'q': 'wa', 'r': 'edft',
    's': 'awedxz', 't': 'rfgy', 'u': 'yhji', 'v': 'cfgb', 'w': 'qase', 'x': 'zsdc',
    'y': 'tghu', 'z': 'asx', ' ': ' ',
}
P_ERR = 0.30   # probability that a keypress slips to an adjacent key

EMIT = np.zeros((N_STATES, N_STATES))
for ch in ALPHABET:
    i = char_index[ch]
    if ch == " ":
        EMIT[i, char_index[" "]] = 1.0          # the space bar is too big to miss
    else:
        neighbours = [c for c in QWERTY_NEIGHBOURS[ch] if c in char_index]
        EMIT[i, i] = 1 - P_ERR                  # usually the right key
        for c in neighbours:                    # otherwise a neighbouring key, uniformly
            EMIT[i, char_index[c]] = P_ERR / len(neighbours)

print(f"\nKeyboard fault model: P(slip to an adjacent key) = {P_ERR:.0%}")

Estimating the HMM from REAL text
Corpus: 593 real newsgroup posts, 683,291 characters after cleaning

What the model learned from real English (top successor of each letter):
  after    'q': 'u'  (p = 0.81)
  after    't': 'h'  (p = 0.25)
  after    's': ' '  (p = 0.39)
  after    'i': 'n'  (p = 0.24)
  after  space: 't'  (p = 0.14)

Keyboard fault model: P(slip to an adjacent key) = 30%


In [7]:
# Corrupt a REAL sentence and try to recover it. The sentence below was written by a real
# person in a real newsgroup post — we pick it from the corpus rather than composing one.
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", raw_text) if 60 < len(s.strip()) < 110]
true_sentence = re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", sentences[2].lower())).strip()

# Inject the keyboard noise. Randomness is the LESSON here: it creates the noisy channel
# whose effect we are about to undo. The underlying sentence is real, not invented.
rng = np.random.default_rng(7)
observed = []
for ch in true_sentence:
    if ch != " " and rng.random() < P_ERR:
        observed.append(str(rng.choice(list(QWERTY_NEIGHBOURS[ch]))))
    else:
        observed.append(ch)
observed_sentence = "".join(observed)

print("=" * 60)
print("A real sentence, typed on a faulty keyboard")
print("=" * 60)
print(f"\n  intended (ground truth): {true_sentence}")
print(f"  observed  (what we see): {observed_sentence}")

A real sentence, typed on a faulty keyboard

  intended (ground truth): however i have been selected to be an exchange student at louisiana state uni
  observed  (what we see): howwvsr i hage been sekrfred to bd qn wxfnznge syudent st oouisians etwye uni


### Viterbi in log space — and why that change is necessary

The `viterbi()` function in Part 3 multiplies probabilities together. Over a three-step
weather sequence that is fine. Over a 77-character sentence with 27 states it is not: the
best-path probability drops below `1e-100` and eventually underflows to exactly `0.0`, at
which point every path ties and the algorithm returns nonsense.

The fix is standard and exact: **add logarithms instead of multiplying probabilities**.
`log(a × b) = log(a) + log(b)`, so the arg-max is identical while the numbers stay in a range
floating point can represent. The version below is the same algorithm as Part 3 — initialise,
recurse with `max` plus a backpointer, then backtrack — written with NumPy arrays and logs.

In [8]:
# The same Viterbi algorithm as Part 3, in log space and vectorised over states.
# Why rewrite it: 77 characters x 27 states would underflow float64 if we multiplied raw
# probabilities, and NumPy lets us score all 27 candidate predecessors in one operation.
def viterbi_log(obs_indices, initial, transition, emission):
    """Most likely hidden-state path, computed with log-probabilities.

    obs_indices : list[int]   observed symbols, as alphabet indices
    initial     : (N,) array  P(first state)
    transition  : (N,N) array P(next state | current state)
    emission    : (N,N) array P(observed symbol | state)
    Returns the best state path as a list of indices.
    """
    log_trans = np.log(transition)
    log_emit = np.log(emission + 1e-300)          # guard against log(0) for impossible slips
    T = len(obs_indices)

    # Initialisation: log P(state) + log P(first observation | state)
    delta = np.log(initial) + log_emit[:, obs_indices[0]]
    backpointer = np.zeros((T, len(initial)), dtype=int)

    # Recursion: for each candidate state, keep the single best predecessor (max, not sum)
    for t in range(1, T):
        scores = delta[:, None] + log_trans       # (prev_state, next_state)
        backpointer[t] = scores.argmax(axis=0)    # remember which predecessor won
        delta = scores.max(axis=0) + log_emit[:, obs_indices[t]]

    # Backtrack from the best final state
    path = [int(delta.argmax())]
    for t in range(T - 1, 0, -1):
        path.insert(0, int(backpointer[t, path[0]]))
    return path


obs_indices = [char_index[c] for c in observed_sentence]
decoded_path = viterbi_log(obs_indices, INITIAL, TRANS, EMIT)
decoded_sentence = "".join(ALPHABET[i] for i in decoded_path)

# Measure honestly: character accuracy before and after decoding.
acc_observed = np.mean([a == b for a, b in zip(observed_sentence, true_sentence)])
acc_decoded = np.mean([a == b for a, b in zip(decoded_sentence, true_sentence)])

print("=" * 60)
print("Viterbi decoding of real text")
print("=" * 60)
print(f"\n  intended : {true_sentence}")
print(f"  observed : {observed_sentence}")
print(f"  decoded  : {decoded_sentence}")
print()
print(f"  Character accuracy BEFORE decoding : {acc_observed:.1%}")
print(f"  Character accuracy AFTER  decoding : {acc_decoded:.1%}")
print(f"  Characters repaired: "
      f"{int(round((acc_decoded - acc_observed) * len(true_sentence)))} of {len(true_sentence)}")

Viterbi decoding of real text

  intended : however i have been selected to be an exchange student at louisiana state uni
  observed : howwvsr i hage been sekrfred to bd qn wxfnznge syudent st oouisians etwye uni
  decoded  : howaver i hage been selefred to be an wsthange student st oouisians etwhe uni

  Character accuracy BEFORE decoding : 74.0%
  Character accuracy AFTER  decoding : 83.1%
  Characters repaired: 7 of 77


### Reading the result honestly

Viterbi does **not** recover the sentence perfectly, and it should not be expected to. The
model only knows about *pairs* of adjacent characters — it has no vocabulary, no words, no
grammar. Where the letter-pair statistics of real English are strong (`q`→`u`, `th`, `ing`,
`ed `) it repairs the text; where a corrupted string is itself a plausible English letter
sequence, it has no basis to prefer the truth.

That partial success is exactly what a real noisy-channel model delivers, and it is the
reason production systems stack a word-level or neural language model on top of this idea
rather than stopping here. A demo built on invented letter statistics could have been tuned
to reach 100% and would have taught you the opposite lesson.

## Summary

### Key Concepts:
1. **HMM Components**: States, observations, transitions, emissions
2. **Forward Algorithm**: Compute probability of observation sequence
3. **Viterbi Algorithm**: Find most likely hidden state sequence
4. **Log-space decoding**: why long sequences need logs instead of raw probabilities
5. **Parameters from real data**: transition probabilities counted from a real English corpus
   rather than chosen by hand
6. **Applications**: Speech recognition, POS tagging, noisy-channel decoding, sequence prediction

### Applications:
- Natural language processing (POS tagging, NER)
- Speech recognition
- Bioinformatics (gene prediction)
- Time series prediction

**Reference:** Course 02, Unit 3: "Working with Hidden Markov Models (HMMs)" and "Implementing Viterbi algorithm for sequence decoding"


## 📚 References

1. Viterbi, A. J. (1967). *Error Bounds for Convolutional Codes and an Asymptotically Optimum Decoding Algorithm*. IEEE Transactions on Information Theory 13(2), 260-269. (origin of the Viterbi algorithm)
2. Rabiner, L. R. (1989). *A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition*. Proceedings of the IEEE 77(2), 257-286. (the classic HMM tutorial)
3. Jurafsky, D., & Martin, J. H. (2024 draft). *Speech and Language Processing* (3rd ed.), Appendix on Hidden Markov Models. <https://web.stanford.edu/~jurafsky/slp3/>